In [0]:
%python
%pip install statsmodels

In [0]:
%python
%restart_python

In [0]:
%python
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

df = spark.table("marketing_analytics.gold.ab_test_results").toPandas().set_index("test_group")

n_ad,  x_ad  = int(df.loc["ad",  "users"]), int(df.loc["ad",  "conversions"])
n_psa, x_psa = int(df.loc["psa", "users"]), int(df.loc["psa", "conversions"])

p_ad, p_psa = x_ad / n_ad, x_psa / n_psa

z, p_value = proportions_ztest([x_ad, x_psa], [n_ad, n_psa])

ci_low, ci_high = confint_proportions_2indep(x_ad, n_ad, x_psa, n_psa, compare="diff")

print(f"ad  rate: {p_ad:.4%}")
print(f"psa rate: {p_psa:.4%}")
print(f"absolute lift: {(p_ad - p_psa):.4%}")
print(f"relative lift: {(p_ad / p_psa - 1):.1%}")
print(f"z = {z:.3f}, p-value = {p_value:.2e}")
print(f"95% CI for difference: [{ci_low:.4%}, {ci_high:.4%}]")

In [0]:
%python
cost_per_impression = 0.01
value_per_conversion = 20

impr_ad = spark.sql("""
  SELECT SUM(total_ads) AS s
  FROM marketing_analytics.silver.ab_test_clean
  WHERE test_group = 'ad'
""").collect()[0]["s"]

extra_conv      = (p_ad - p_psa) * n_ad
extra_conv_low  = ci_low  * n_ad
extra_conv_high = ci_high * n_ad

cost    = impr_ad * cost_per_impression
revenue = extra_conv * value_per_conversion
net     = revenue - cost
roi     = net / cost
breakeven_value = cost / extra_conv

print(f"extra conversions: {extra_conv:,.0f} (95% CI {extra_conv_low:,.0f} to {extra_conv_high:,.0f})")
print(f"revenue: ${revenue:,.0f}  (CI ${extra_conv_low*value_per_conversion:,.0f} to ${extra_conv_high*value_per_conversion:,.0f})")
print(f"cost: ${cost:,.0f}")
print(f"net: ${net:,.0f}  |  ROI: {roi:.1%}")
print(f"break-even value per conversion: ${breakeven_value:,.2f}")

In [0]:
%python
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

f = spark.table("marketing_analytics.gold.ab_test_by_frequency").toPandas()

rows = []
for bucket, g in f.groupby("ad_bucket"):
    g = g.set_index("test_group")
    n_ad,  x_ad  = int(g.loc["ad",  "users"]), int(g.loc["ad",  "conversions"])
    n_psa, x_psa = int(g.loc["psa", "users"]), int(g.loc["psa", "conversions"])
    lift = x_ad / n_ad - x_psa / n_psa
    lo, hi = confint_proportions_2indep(x_ad, n_ad, x_psa, n_psa, compare="diff")
    _, p = proportions_ztest([x_ad, x_psa], [n_ad, n_psa])
    rows.append({
        "bucket": bucket, "psa_users": n_psa, "psa_conversions": x_psa,
        "lift_pts": lift * 100, "ci_low_pts": lo * 100,
        "ci_high_pts": hi * 100, "p_value": p,
    })

print(pd.DataFrame(rows).round(3).to_string(index=False))

In [0]:
%python
def lift_table(table, group_col):
    f = spark.table(table).toPandas()
    rows = []
    for key, g in f.groupby(group_col):
        g = g.set_index("test_group")
        n_ad,  x_ad  = int(g.loc["ad",  "users"]), int(g.loc["ad",  "conversions"])
        n_psa, x_psa = int(g.loc["psa", "users"]), int(g.loc["psa", "conversions"])
        lo, hi = confint_proportions_2indep(x_ad, n_ad, x_psa, n_psa, compare="diff")
        _, p = proportions_ztest([x_ad, x_psa], [n_ad, n_psa])
        rows.append({group_col: key, "lift_pts": (x_ad/n_ad - x_psa/n_psa)*100,
                     "ci_low_pts": lo*100, "ci_high_pts": hi*100, "p_value": p})
    return pd.DataFrame(rows).round(3)

print(lift_table("marketing_analytics.gold.ab_test_by_day", "day_num"))